# Universidad del Valle de Guatemala
## Departamento de Ciencias de la Computación / Matemática
### Modelación y Simulación 2026 - Laboratorio 04

**Integrantes:**
* Erick Guerra (`eagi578@gmail.com`)
* Fabian Morales (`fabimoradav2004@gmail.com`)
* Diego Patzan (`diegopatzan24@gmail.com`)

**Fecha:** 1 al 2 de septiembre de 2026

---

## 0. Importación de Librerías y Configuración
En este notebook se utiliza la librería `PuLP` para la formulación y resolución de problemas de programación lineal (continua y entera).

In [ ]:
import pulp
import pandas as pd

print(f"Versión de PuLP cargada: {pulp.__version__}")

## Problema 2: Modelo de Producción, Períodos Múltiples (ACME Manufacturing Company)

### a) Formulación del Modelo de Programación Lineal

#### Variables de Decisión:
* $x_t \ge 0$: Cantidad de ventanas a producir en el mes $t \in \{1, 2, 3, 4, 5, 6\}$.
* $I_t \ge 0$: Cantidad de ventanas en inventario al final del mes $t \in \{1, 2, 3, 4, 5, 6\}$.

#### Función Objetivo:
Minimizar los costos totales de producción e inventario durante los 6 meses:
$$\min Z = \sum_{t=1}^{6} \left( c_t \cdot x_t + h_t \cdot I_t \right)$$

donde los costos unitarios de producción son $c = [50, 45, 55, 52, 48, 50]$ y los costos unitarios de almacenamiento son $h = [8, 10, 10, 10, 8, 8]$.

#### Restricciones:
1. **Balance de Inventario por Período ($I_{t} = I_{t-1} + x_t - d_t$ con $I_0 = 0$):**
   * Mes 1: $I_1 = x_1 - 180$
   * Mes 2: $I_2 = I_1 + x_2 - 250$
   * Mes 3: $I_3 = I_2 + x_3 - 190$
   * Mes 4: $I_4 = I_3 + x_4 - 140$
   * Mes 5: $I_5 = I_4 + x_5 - 220$
   * Mes 6: $I_6 = I_5 + x_6 - 250$
2. **Capacidad Máxima de Producción Mensual:**
   $$x_t \le 225, \quad \forall t \in \{1..6\}$$
3. **No Negatividad:**
   $$x_t \ge 0, \quad I_t \ge 0, \quad \forall t \in \{1..6\}$$

In [ ]:
meses = list(range(1, 7))
demanda = {1: 180, 2: 250, 3: 190, 4: 140, 5: 220, 6: 250}
costo_prod = {1: 50, 2: 45, 3: 55, 4: 52, 5: 48, 6: 50}
costo_inv = {1: 8, 2: 10, 3: 10, 4: 10, 5: 8, 6: 8}
capacidad_max = 225

# Modelo Continuo
prob_cont = pulp.LpProblem("ACME_Produccion_Continuo", pulp.LpMinimize)
x_cont = pulp.LpVariable.dicts("Prod", meses, lowBound=0, cat=pulp.LpContinuous)
I_cont = pulp.LpVariable.dicts("Inv", meses, lowBound=0, cat=pulp.LpContinuous)

prob_cont += pulp.lpSum([costo_prod[t] * x_cont[t] + costo_inv[t] * I_cont[t] for t in meses])

for t in meses:
    prob_cont += x_cont[t] <= capacidad_max, f"Capacidad_Mes_{t}"
    if t == 1:
        prob_cont += I_cont[t] == x_cont[t] - demanda[t], f"Balance_Mes_{t}"
    else:
        prob_cont += I_cont[t] == I_cont[t-1] + x_cont[t] - demanda[t], f"Balance_Mes_{t}"

prob_cont.solve(pulp.PULP_CBC_CMD(msg=0))

# Modelo Entero
prob_ent = pulp.LpProblem("ACME_Produccion_Entero", pulp.LpMinimize)
x_ent = pulp.LpVariable.dicts("Prod", meses, lowBound=0, cat=pulp.LpInteger)
I_ent = pulp.LpVariable.dicts("Inv", meses, lowBound=0, cat=pulp.LpInteger)

prob_ent += pulp.lpSum([costo_prod[t] * x_ent[t] + costo_inv[t] * I_ent[t] for t in meses])

for t in meses:
    prob_ent += x_ent[t] <= capacidad_max, f"Capacidad_Mes_{t}"
    if t == 1:
        prob_ent += I_ent[t] == x_ent[t] - demanda[t], f"Balance_Mes_{t}"
    else:
        prob_ent += I_ent[t] == I_ent[t-1] + x_ent[t] - demanda[t], f"Balance_Mes_{t}"

prob_ent.solve(pulp.PULP_CBC_CMD(msg=0))

# Presentación de Resultados en Tabla
df_res2 = pd.DataFrame({
    'Mes': meses,
    'Demanda': [demanda[t] for t in meses],
    'Producción (x_t)': [x_cont[t].varValue for t in meses],
    'Inventario Final (I_t)': [I_cont[t].varValue for t in meses],
    'Costo Prod ($)': [x_cont[t].varValue * costo_prod[t] for t in meses],
    'Costo Inv ($)': [I_cont[t].varValue * costo_inv[t] for t in meses]
})
df_res2['Costo Total ($)'] = df_res2['Costo Prod ($)'] + df_res2['Costo Inv ($)']

print("==========================================================")
print("PROBLEMA 2: MODELO DE PRODUCCIÓN E INVENTARIO (ACME)")
print("==========================================================")
print(f"Estado Solución: {pulp.LpStatus[prob_cont.status]}")
print(f"Costo Total Óptimo: ${pulp.value(prob_cont.objective):,.2f}\n")
print(df_res2.to_string(index=False))

print("\n--- DIAGRAMA DE FLUJO DE PRODUCCIÓN E INVENTARIO ---")
print("I0 = 0")
for t in meses:
    prev_inv = 0 if t == 1 else I_cont[t-1].varValue
    print(f"Mes {t}: Inv_Inic ({prev_inv:.0f}) + Prod ({x_cont[t].varValue:.0f}) - Demanda ({demanda[t]}) ==> Inv_Fin ({I_cont[t].varValue:.0f})")

print("\n--- COMPARACIÓN CON MODELO ENTERO (Inciso c) ---")
print(f"Costo Óptimo Modelo Entero: ${pulp.value(prob_ent.objective):,.2f}")
print("¿Se obtiene la misma solución óptima? SÍ, la solución continua resulta ser naturally entera.")

## Problema 3: Modelo de Asignación de Horarios (Autobuses Ciudad de Guatemala)

### a) Formulación del Modelo de Programación Lineal

#### Variables de Decisión:
* $x_i \ge 0$: Cantidad de autobuses que inician operaciones en el turno $i \in \{1, 2, 3, 4, 5, 6\}$. Cada autobús labora 8 horas continuas (cubre 2 tramos consecutivos de 4 horas).

#### Función Objetivo:
Minimizar la cantidad total de autobuses necesarios al día:
$$\min Z = \sum_{i=1}^{6} x_i$$

#### Restricciones de Cobertura por Tramo Horario de 4 Horas:
* Tramo 1 (00:00 - 04:00, requeridos: 4): $x_6 + x_1 \ge 4$
* Tramo 2 (04:00 - 08:00, requeridos: 8): $x_1 + x_2 \ge 8$
* Tramo 3 (08:00 - 12:00, requeridos: 10): $x_2 + x_3 \ge 10$
* Tramo 4 (12:00 - 16:00, requeridos: 7): $x_3 + x_4 \ge 7$
* Tramo 5 (16:00 - 20:00, requeridos: 12): $x_4 + x_5 \ge 12$
* Tramo 6 (20:00 - 00:00, requeridos: 4): $x_5 + x_6 \ge 4$
* No negatividad: $x_i \ge 0, \quad \forall i \in \{1..6\}$

In [ ]:
turnos = list(range(1, 7))
horarios_turnos = {
    1: "00:00 A.M. a 07:59 A.M.",
    2: "04:00 A.M. a 11:59 A.M.",
    3: "08:00 A.M. a 03:59 P.M.",
    4: "12:00 P.M. a 07:59 P.M.",
    5: "04:00 P.M. a 11:59 P.M.",
    6: "08:00 P.M. a 03:59 A.M."
}

tramos = {
    1: ("12:00 A.M. - 04:00 A.M.", 4, [6, 1]),
    2: ("04:00 A.M. - 08:00 A.M.", 8, [1, 2]),
    3: ("08:00 A.M. - 12:00 P.M.", 10, [2, 3]),
    4: ("12:00 P.M. - 04:00 P.M.", 7, [3, 4]),
    5: ("04:00 P.M. - 08:00 P.M.", 12, [4, 5]),
    6: ("08:00 P.M. - 12:00 A.M.", 4, [5, 6]),
}

# Modelo Continuo
prob3_cont = pulp.LpProblem("Asignacion_Buses_Continuo", pulp.LpMinimize)
x3_cont = pulp.LpVariable.dicts("Buses", turnos, lowBound=0, cat=pulp.LpContinuous)
prob3_cont += pulp.lpSum([x3_cont[i] for i in turnos])

for t_id, (nombre_tramo, dem, t_cubren) in tramos.items():
    prob3_cont += pulp.lpSum([x3_cont[i] for i in t_cubren]) >= dem, f"Demanda_Tramo_{t_id}"

prob3_cont.solve(pulp.PULP_CBC_CMD(msg=0))

# Modelo Entero
prob3_ent = pulp.LpProblem("Asignacion_Buses_Entero", pulp.LpMinimize)
x3_ent = pulp.LpVariable.dicts("Buses", turnos, lowBound=0, cat=pulp.LpInteger)
prob3_ent += pulp.lpSum([x3_ent[i] for i in turnos])

for t_id, (nombre_tramo, dem, t_cubren) in tramos.items():
    prob3_ent += pulp.lpSum([x3_ent[i] for i in t_cubren]) >= dem, f"Demanda_Tramo_{t_id}"

prob3_ent.solve(pulp.PULP_CBC_CMD(msg=0))

df_turnos3 = pd.DataFrame({
    'Turno': turnos,
    'Horario': [horarios_turnos[i] for i in turnos],
    'Buses a Iniciar (Continuo)': [x3_cont[i].varValue for i in turnos],
    'Buses a Iniciar (Entero)': [x3_ent[i].varValue for i in turnos]
})

print("==========================================================")
print("PROBLEMA 3: ASIGNACIÓN DE HORARIOS DE BUSES (GUATEMALA)")
print("==========================================================")
print(f"Total Mínimo de Autobuses Requeridos: {pulp.value(prob3_cont.objective):.0f} buses\n")
print(df_turnos3.to_string(index=False))

df_cobertura3 = pd.DataFrame({
    'Tramo': list(tramos.keys()),
    'Horario': [tramos[k][0] for k in tramos],
    'Requeridos': [tramos[k][1] for k in tramos],
    'Asignados': [sum(x3_cont[i].varValue for i in tramos[k][2]) for k in tramos]
})
df_cobertura3['Excedente'] = df_cobertura3['Asignados'] - df_cobertura3['Requeridos']

print("\n--- COBERTURA DE DEMANDA POR TRAMO HORARIO ---")
print(df_cobertura3.to_string(index=False))